# Ollama and the harness recipe

Goal: construct `Agent.ollama`, then copy the harness recipe: one provider factory, a Python toolset, an observer, `start`, events, and `result`.

Trust: T1 native provider plus T2 Python ports. Network: construct-only unless local Ollama has `gemma4:26b`.

There is no Python `Harness` type. `Agent` is the composition root. `ollama` stays keyless. The live cell uses `gemma4:26b` (or `OLLAMA_MODEL`) when that model is installed. It skips if Ollama is down or the model is missing.


In [ ]:
import finstack_ai

agent = await finstack_ai.Agent.ollama(
    "http://127.0.0.1:11434",
    "gemma4:26b",
    "Answer concisely.",
)
print(agent.compact_capability_catalog())

In [ ]:
import os
from typing import Any

from pydantic import BaseModel

from _support import ollama_installed, ollama_live

import finstack_ai


class Answer(BaseModel):
    answer: int


@finstack_ai.tool
def add(left: int, right: int) -> Answer:
    """Add two integers."""
    return Answer(answer=left + right)


tools = finstack_ai.pydantic_toolset(
    add,
    component="notebook.toolset.ollama",
    name="math",
)
seen: list[dict[str, Any]] = []


async def observe(batch: list[dict[str, Any]]) -> None:
    seen.extend(batch)


observer = finstack_ai.PythonObserver(
    observe,
    component="notebook.observer.ollama",
    payload_mode="redacted",
)
base_url = "http://127.0.0.1:11434"
model = os.environ.get("OLLAMA_MODEL") or "gemma4:26b"
installed = ollama_installed(base_url)
agent = await finstack_ai.Agent.ollama(
    base_url,
    model,
    "Answer concisely.",
    toolsets=[tools],
    observers=[observer],
)
print(model)
print(agent.compact_capability_catalog())

if ollama_live(base_url, model):
    run = agent.start("Add 20 and 22. Are you gemma?")
    batches = [batch async for batch in run.events()]
    result = await run.result()
    print(result.text)
    print(batches[-1].events()[-1].kind)
elif installed is None:
    print("skipped: Ollama not reachable")
else:
    listed = ", ".join(installed) if installed else "(none)"
    print(f"skipped: {model} not installed; installed: {listed}")

Copy this recipe for any linked provider: choose `openai`, `anthropic`, or `ollama`, attach trusted Python ports, call `start`, iterate `events()`, then `await result()`. Use `run()` when you do not need the live handle. Call `Run.cancel()` to cancel; dropping the handle does not.
